In [7]:
import pandas as pd
import numpy as np
import csv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(
    '../datos/dataset_nlp_utf16_final.csv', 
    encoding='utf-16', 
    sep=',', 
    quoting=csv.QUOTE_ALL, 
    escapechar='\\'
)

# Unificación de textos para comparar
# Esto evita que un NaN destruya el texto de la otra columna al concatenarlos (sumarlos)
# Ejemplo de error anterior ("Conocimientos en Excel" + NaN = NaN)
df['cargo'] = df['cargo'].fillna('')
df['funciones'] = df['funciones'].fillna('')
df['conocimientos'] = df['conocimientos'].fillna('')
df['experiencia'] = df['experiencia'].fillna('')
df['experiencia_cv'] = df['experiencia_cv'].fillna('')
df['estudios_cv'] = df['estudios_cv'].fillna('')

# Juntamos lo que pide la empresa
df['texto_oferta'] = df['cargo'] + " " + df['funciones'] + " " + df['conocimientos'] + " " + df['experiencia']
# Juntamos lo que ofrece el candidato
df['texto_candidato'] = df['experiencia_cv'] + " " + df['estudios_cv']

# Trabajaremos los textos en minúsculas
df['texto_oferta'] = df['texto_oferta'].fillna('').str.lower()
df['texto_candidato'] = df['texto_candidato'].fillna('').str.lower()

print(f"Total de registros listos para análisis: {df.shape[0]}")

Total de registros listos para análisis: 220


In [8]:
# Trabajo adaptado para texto en español
import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')  # Descarga del paquete de stop words
stop_words_es = stopwords.words('spanish')  # Crea una lista con las palabras vacías en español

print("Vectorizando textos con TF-IDF en español.")

vectorizador = TfidfVectorizer(max_features=5000, stop_words=stop_words_es)

# Ajusta el vocabulario usando tanto las ofertas como los CVs
corpus_completo = pd.concat([df['texto_oferta'], df['texto_candidato']])
vectorizador.fit(corpus_completo)

# Transforma textos a matrices numéricas
tfidf_ofertas = vectorizador.transform(df['texto_oferta'])
tfidf_candidatos = vectorizador.transform(df['texto_candidato'])

# Calcula la similitud del coseno fila por fila
similitudes = []
for i in range(df.shape[0]):
    # Calcula qué tan cerca está el vector del CV al vector de la Oferta (1 = Idéntico, 0 = Ninguna similitud)
    similitud = cosine_similarity(tfidf_ofertas[i], tfidf_candidatos[i])[0][0]
    similitudes.append(similitud)

df['score_similitud'] = similitudes
print("Similitud calculada. Muestra de los primeros 5 scores:")
print(df[['target_elegibilidad', 'score_similitud']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Vectorizando textos con TF-IDF en español.
Similitud calculada. Muestra de los primeros 5 scores:
   target_elegibilidad  score_similitud
0                    1         0.064945
1                    1         0.052568
2                    1         0.033694
3                    1         0.000000
4                    1         0.211424


In [11]:
# Calcula cantidades y porcentajes
total_filas = len(df)
filas_cero = (df['score_similitud'] == 0.0).sum()
porcentaje_ceros = (filas_cero / total_filas) * 100

print(f"Total de registros evaluados: {total_filas}")
print(f"Registros con similitud exacta de 0.0: {filas_cero}")
print(f"Porcentaje de ceros en el dataset: {porcentaje_ceros:.2f}%\n")

Total de registros evaluados: 220
Registros con similitud exacta de 0.0: 54
Porcentaje de ceros en el dataset: 24.55%

